In [ ]:
# Full-Mamba HADA — PhysioNet 109-subject source-only LOSO, single GPU, batch 144.
# Run 28 Kaggle jobs with JOB_INDEX=0..27; each job evaluates 4 target folds sequentially on GPU 0.
import codecs
import math
import os
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

# ---------------- Job configuration ----------------
JOB_INDEX = 0          # Change to 0, 1, ..., 27 in separate Kaggle jobs.
FOLDS_PER_JOB = 4     # Sized to stay near the workload of the older 10-fold/175-epoch jobs.
MAX_EPOCHS = 75
TRAINING_MODE = 'source_only'  # Dedicated source-only baseline on this branch.
if TRAINING_MODE not in {'autodial', 'source_only'}:
    raise ValueError("TRAINING_MODE must be 'autodial' or 'source_only'")
USE_RA = False        # 64-channel Riemannian means are too costly for this full benchmark.
IM_TTA_STEPS = 0      # Keep the large-cohort run within the Kaggle time budget.

ALL_SUBJECTS = list(range(1, 110))
TOTAL_JOBS = math.ceil(len(ALL_SUBJECTS) / FOLDS_PER_JOB)
if not 0 <= JOB_INDEX < TOTAL_JOBS:
    raise ValueError(f'JOB_INDEX must be in 0..{TOTAL_JOBS - 1}')
start = JOB_INDEX * FOLDS_PER_JOB
TARGET_SUBJECTS = ALL_SUBJECTS[start:start + FOLDS_PER_JOB]
JOB_TAG = f's{TARGET_SUBJECTS[0]:03d}_s{TARGET_SUBJECTS[-1]:03d}'

DATASET = 'physionet'
BRANCH = (
    'feature/hada-autodial-zhou-mi-preprocessing'
    if TRAINING_MODE == 'autodial'
    else 'feature/hada-full-mamba-no-da'
)
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
RUN_TAG = f'{DATASET}109_{TRAINING_MODE}_single_gpu_bs144_ep{MAX_EPOCHS}_{JOB_TAG}'
REPO_PATH = Path(f'/kaggle/working/tcformer-{RUN_TAG}')
MNE_DATA = Path('/kaggle/working/mne_data')
TRAIN_LOG = Path(f'/kaggle/working/{RUN_TAG}.log')
RESULT_ARCHIVE = Path(f'/kaggle/working/{RUN_TAG}_results')


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return

    process = subprocess.Popen(
        command, cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
    )
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f'[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Training exited with code {return_code}. Full log: {log_path}')


def run_with_retries(command, attempts=5, cwd=None):
    for attempt in range(1, attempts + 1):
        try:
            run(command, cwd=cwd)
            return
        except subprocess.CalledProcessError:
            if attempt == attempts:
                raise
            delay = min(15 * attempt, 60)
            print(f'Command failed (attempt {attempt}/{attempts}); retrying in {delay}s ...', flush=True)
            time.sleep(delay)


# ---------------- Clone the requested branch ----------------
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

# The base branch intentionally used only S001-S020. Expand the source pool to all 109
# subjects while keeping TARGET_SUBJECTS as the folds evaluated by this shard.
physionet_path = REPO_PATH / 'datamodules/physionet.py'
physionet_text = physionet_path.read_text(encoding='utf-8')
replacements = {
    'First-20-subject PhysioNet': 'Full-109-subject PhysioNet',
    'Each of the 19': 'Each of the 108',
    'all_subject_ids = list(range(1, 21))': 'all_subject_ids = list(range(1, 110))',
    'target subject must be in 1..20': 'target subject must be in 1..109',
    'Materializing all 20 subjects': 'Materializing all 109 subjects',
    'did not materialize subjects 1..20': 'did not materialize subjects 1..109',
    'if len(source_ids) != 19': 'if len(source_ids) != 108',
}
for old, new in replacements.items():
    if old not in physionet_text:
        raise RuntimeError(f'Expected PhysioNet source marker is missing: {old!r}')
    physionet_text = physionet_text.replace(old, new)
physionet_path.write_text(physionet_text, encoding='utf-8')
assert 'all_subject_ids = list(range(1, 110))' in physionet_text
assert 'if len(source_ids) != 108' in physionet_text

pipeline_text = (REPO_PATH / 'train_pipeline.py').read_text(encoding='utf-8')
model_text = (REPO_PATH / 'models/hada_tcformer.py').read_text(encoding='utf-8')
assert 'ModelCheckpoint' in pipeline_text and 'monitor="val_acc"' in pipeline_text
if TRAINING_MODE == 'autodial':
    assert 'class AutoDIALFeatureNorm' in model_text
else:
    assert 'ordinary labeled' in model_text and 'source batch' in model_text
print(f'Verified: 109-subject source pool, {TRAINING_MODE}, and source-validation checkpointing.')

# ---------------- Environment ----------------
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
     '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run([UV, 'pip', 'install', '--python', PYTHON, 'awscli'])

# ---------------- Resumable PhysioNet prefetch ----------------
# The PhysioNet web endpoint can throttle long Kaggle downloads or return HTTP 502.
# Prefetch only the six motor-imagery runs from PhysioNet's public AWS mirror into
# MNE's exact EEGBCI cache layout. Re-running this cell resumes/skips completed EDFs.
eegbci_cache = MNE_DATA / 'MNE-eegbci-data/files/eegmmidb/1.0.0'
eegbci_cache.mkdir(parents=True, exist_ok=True)
AWS = REPO_PATH / '.venv/bin/aws'
aws_sync = [
    AWS, 's3', 'sync',
    's3://physionet-open/eegmmidb/1.0.0/', eegbci_cache,
    '--no-sign-request', '--only-show-errors', '--exclude', '*',
]
for run_id in (4, 6, 8, 10, 12, 14):
    aws_sync.extend(['--include', f'S*/S*R{run_id:02d}.edf'])
run_with_retries(aws_sync, attempts=5)
cached_edfs = list(eegbci_cache.glob('S*/S*R*.edf'))
expected_edfs = len(ALL_SUBJECTS) * 6
if len(cached_edfs) != expected_edfs:
    raise RuntimeError(f'PhysioNet prefetch incomplete: {len(cached_edfs)}/{expected_edfs} EDF files')
print(f'PhysioNet AWS prefetch complete: {len(cached_edfs)}/{expected_edfs} EDF files')

# ---------------- 75-epoch full-cohort override ----------------
if TRAINING_MODE == 'autodial':
    mode_override = (
        "c['domain_adaptation']=True; "
        "c['model_kwargs']['feature_alignment']='autodial'; "
    )
else:
    # Pure source-only: no target batches during training, no RA, and no IM-TTA.
    mode_override = (
        "c['domain_adaptation']=False; "
        "c['model_kwargs'].pop('feature_alignment', None); "
        "c['model_kwargs']['adversarial_weight']=0.0; "
        "c['model_kwargs']['mmd_weight']=0.0; "
        "c['model_kwargs']['temporal_mmd_weight']=0.0; "
    )
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    f"c['max_epochs_loso']={MAX_EPOCHS}; "
    "c['log_every_n_epochs']=10; c['enable_progress_bar']=False; "
    "c['preprocessing']['physionet']['trial_duration']=4.0; "
    "c['preprocessing']['physionet']['batch_size']=144; "
    "c['preprocessing']['physionet']['accumulate_grad_batches']=1; "
    "c['preprocessing']['physionet']['num_workers']=2; "
    f"c['preprocessing']['physionet']['riemannian_alignment']={USE_RA!r}; "
    f"c['preprocessing']['physionet'].setdefault('model_overrides', {{}})['im_tta_steps']={IM_TTA_STEPS}; "
    f"{mode_override}"
    "c['model_kwargs']['log_every_n_batches']=0; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); "
     "print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
     "print('GPU:', torch.cuda.get_device_name(0))"])

# ---------------- Train / test this shard ----------------
MNE_DATA.mkdir(parents=True, exist_ok=True)
results_dir = REPO_PATH / 'results'
if results_dir.exists():
    shutil.rmtree(results_dir)
environment = os.environ.copy()
environment.update({
    'PYTHONUNBUFFERED': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'MPLBACKEND': 'Agg',
    'MNE_DATA': str(MNE_DATA),
    'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA),
})
print(f'===== {TRAINING_MODE.upper()} FULL-MAMBA HADA | PHYSIONET 109-SUBJECT LOSO | SINGLE GPU 0 ' 
      f'| JOB {JOB_INDEX + 1}/{TOTAL_JOBS} | TARGETS={TARGET_SUBJECTS} | BATCH=144 | ACCUM=1 ' 
      f'| 75 EPOCHS | 4 CLASSES | 64 CH =====', flush=True)
if TRAINING_MODE == 'autodial':
    print('Protocol: each fold trains on 108 labeled source subjects; the held-out target is unlabeled during AutoDIAL training and used for final scoring only.', flush=True)
else:
    print('Protocol: each fold trains only on 108 labeled source subjects; target EEG and labels are excluded from training and adaptation, then used only for final scoring.', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', DATASET,
     '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment, stream=True,
    log_path=TRAIN_LOG)

if TRAINING_MODE == 'autodial':
    log_text = TRAIN_LOG.read_text(encoding='utf-8', errors='replace')
    alpha_prefix = 'AutoDIAL alpha per feature group: '
    alphas = [line.split(alpha_prefix, 1)[1].strip() for line in log_text.splitlines() if alpha_prefix in line]
    for subject_id, alpha in zip(TARGET_SUBJECTS, alphas):
        print(f'S{subject_id:03d} final alpha (CNN groups 1-3, SSM group): {alpha}')

archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Completed targets:', TARGET_SUBJECTS)
print('Kaggle output archive:', archive)
print('Kaggle output log:', TRAIN_LOG)
